# OR7310: wind-farm planning with AMPL
Use **Region A (70 GWh)** to check your approach, then **Region B (300 GWh)** for the main project. This notebook implements the formulation in the handout. AMPL handles the model; Gurobi is the optimization solver called by AMPL.

Open the `provided materials` folder in VS Code and select a Python kernel. Run cells in order. Start with `CASE = "tiny"`, then change it to `"regionA"` or `"regionB"` and rerun from the configuration cell. See `README.md` for setup and dataset details.

In [ ]:
%pip install amplpy pandas

In [ ]:
from pathlib import Path
import sys
import time
import pandas as pd
from amplpy import AMPL, add_to_path

print("Python kernel:", sys.executable)
BASE = Path.cwd()
assert (BASE / "quota_steiner.mod").exists(), "Open provided materials as your working folder."
from project_helpers import export_and_validate

# Use the AMPL installation supplied for class; change this for your computer.
# If AMPL is already on PATH, set AMPL_FOLDER = None.
AMPL_FOLDER = "/Applications/AMPL"   # Windows example: r"C:\ampl"
if AMPL_FOLDER is not None:
    add_to_path(AMPL_FOLDER)

## Choose an instance
The default run is a four-node example with known optimum 8. Region B contains over 1.34 million undirected edges and needs substantially more memory. Close an existing AMPL instance before changing cases.

`None` leaves time and thread settings at solver defaults (no user-imposed time limit). To make a short trial, set `TIME_LIMIT = 60` or `300`. A time-limited feasible solution is useful even when optimality has not been proved.

In [ ]:
CASE = "tiny"                 # "tiny", "regionA", or "regionB"
TIME_LIMIT = None             # seconds; None means no imposed time limit
THREADS = None                # None means solver default

if "ampl" in globals():
    ampl.close()

assert CASE in {"tiny", "regionA", "regionB"}
data_file = (BASE / "tiny.dat" if CASE == "tiny"
             else BASE / "datasets" / CASE / f"{CASE}.dat")
assert data_file.is_file(), f"Missing supplied data file: {data_file}"
print("AMPL data:", data_file)


## Load the model
`x[u,v]` selects one undirected cable and pays for it once. `z[v]` selects a vertex. `forward[u,v]` and `reverse[u,v]` are artificial connectivity flows, for modeling purposes. Every selected nonroot vertex consumes one unit, including intermediate junctions. The edge-count constraint makes the connected selection a tree.

In [ ]:
model_load_start = time.perf_counter()
ampl = AMPL()
ampl.read(str(BASE / "quota_steiner.mod"))
ampl.read_data(str(data_file))
ampl.option["solver"] = "gurobi"
options = ["mip:bestbound=1", "mipgap=0.0001"]
if TIME_LIMIT is not None:
    options.append(f"timelim={TIME_LIMIT}")
if THREADS is not None:
    options.append(f"threads={THREADS}")
ampl.option["gurobi_options"] = " ".join(options)
print("Solver options:", ampl.option["gurobi_options"])
print("Vertices:", ampl.get_value("card(V)"))
print("Undirected edges:", ampl.get_value("card(E)"))
model_load_seconds = time.perf_counter() - model_load_start
print("Model/data loading seconds:", round(model_load_seconds, 2))


In [ ]:
start = time.perf_counter()
ampl.solve()
solve_seconds = time.perf_counter() - start
print("Status:", ampl.get_value("solve_result"))
print("Message:", ampl.get_value("solve_message"))
print("Solve seconds:", round(solve_seconds, 2))

## Validate and save the selected network
The checker tests connectivity, absence of cycles, inclusion of all fixed vertices, energy quota, integrality, and objective consistency. It rejects a fractional or infeasible result rather than reporting it as a usable plan. A nonzero gap means there is still uncertainty about optimality. Selected-edge and selected-node CSVs and a JSON summary are saved in `results/`.

If the solver reports infeasibility, no solution, or a resource failure, inspect its message before running this cell. The small example should have edges (1,2), (2,3), (2,4), energy 9, and cost 8.
Timing is reported separately for model/data loading and the solve call. The solve call includes AMPL model generation and solver execution; use the solver log for its optimization time. The reported `relative_gap` is a fraction: multiply by 100 to express it as a percentage.


In [ ]:
report = export_and_validate(ampl, BASE / "results", CASE)
report["solve_seconds"] = solve_seconds
report["model_load_seconds"] = model_load_seconds
report["time_limit"] = TIME_LIMIT
report["threads"] = THREADS
report["solver_options"] = ampl.option["gurobi_options"]
import json
(BASE / "results" / f"{CASE}_summary.json").write_text(json.dumps(report, indent=2))
display(pd.Series(report))
display(pd.read_csv(BASE / "results" / f"{CASE}_edges.csv").head(10))

## Your project work
Use the cost-only solution as your baseline. Develop and evaluate your improvement for Part A, then implement the weighted objective and comparisons for Part B.

## Part B: Use the supplied landscape data
Each region folder already contains its landscape data file. `h[u,v]` is the cable impact $h_e=\ell_e s_e$, and `s_turbine[v]` is the turbine score $s_v$.

All required inputs are supplied as AMPL `.dat` files. No preprocessing is needed.


In [ ]:
landscape_file = None
if CASE == "tiny":
    print("Part B data are supplied for regionA and regionB, not the tiny example.")
else:
    landscape_file = BASE / "datasets" / CASE / f"{CASE}_landscape.dat"
    assert landscape_file.is_file(), f"Missing supplied file: {landscape_file}"
    print("Supplied Part B data:", landscape_file)


### Use these inputs in your Part B model
In your own copy of the AMPL model, declare:

```ampl
param h {E} >= 0;
param s_turbine {P} >= 0;
```

After loading the original instance data, load the supplied file with `ampl.read_data(str(landscape_file))`. Its edge indices use exactly the same order as the baseline model. These declarations and the extra data file alone do not change the objective.

Your task is to implement the handout's weighted objective, using `sum {(u,v) in E} h[u,v]*x[u,v] + sum {v in P} s_turbine[v]*z[v]` for landscape impact. Compare the requested weights and report monetary cost and landscape impact separately. 

The supplied `export_and_validate` function checks the cost-only objective. If you change the objective for Part B, you need to adapt its objective-consistency check and bound/gap reporting to the weighted objective, while keeping the tree, fixed-terminal, and energy checks.

You can select the supplied Part B file without solving the baseline again: run the setup and case-selection cells, then the Part B file-selection cell. Create a fresh AMPL object for your modified model, and load your modified `.mod`, the original instance `.dat`, then the supplied landscape `.dat`.


In [ ]:
# Release the AMPL process and its memory when finished.
# Rerun the model-loading cell if you want to solve again.
ampl.close()